In [51]:
import pandas as pd
import numpy as np
import os

# ── VERIFY YOU ARE IN RIGHT FOLDER ───────────────
print("Current folder:", os.getcwd())
print()

# ── LOAD DATASET ──────────────────────────────────
df = pd.read_csv("Dataset.csv")

print("=" * 50)
print("DATASET OVERVIEW")
print("=" * 50)
print(f"Rows         : {df.shape[0]}")
print(f"Columns      : {df.shape[1]}")
print()
print("Column names:")
for i, col in enumerate(df.columns, 1):
    print(f"  {i:2}. {col}")

Current folder: C:\Users\BIT\Desktop\sql project

DATASET OVERVIEW
Rows         : 3900
Columns      : 18

Column names:
   1. Customer ID
   2. Age
   3. Gender
   4. Item Purchased
   5. Category
   6. Purchase Amount (USD)
   7. Location
   8. Size
   9. Color
  10. Season
  11. Review Rating
  12. Subscription Status
  13. Shipping Type
  14. Discount Applied
  15. Promo Code Used
  16. Previous Purchases
  17. Payment Method
  18. Frequency of Purchases


In [52]:
# ── FIRST 5 ROWS ──────────────────────────────────
print("First 5 rows:")
print(df.head())
print()

# ── DATA TYPES ────────────────────────────────────
print("Data types:")
print(df.dtypes)
print()

# ── MISSING VALUES ────────────────────────────────
print("Missing values:")
missing = df.isnull().sum()
print(missing[missing > 0] if missing.sum() > 0
      else " No missing values!")
print()

# ── BASIC STATISTICS ──────────────────────────────
print("Basic statistics:")
print(df.describe())

First 5 rows:
   Customer ID  Age Gender Item Purchased  Category  Purchase Amount (USD)  \
0            1   55   Male         Blouse  Clothing                     53   
1            2   19   Male        Sweater  Clothing                     64   
2            3   50   Male          Jeans  Clothing                     73   
3            4   21   Male        Sandals  Footwear                     90   
4            5   45   Male         Blouse  Clothing                     49   

        Location Size      Color  Season  Review Rating Subscription Status  \
0       Kentucky    L       Gray  Winter            3.1                 Yes   
1          Maine    L     Maroon  Winter            3.1                 Yes   
2  Massachusetts    S     Maroon  Spring            3.1                 Yes   
3   Rhode Island    M     Maroon  Spring            3.5                 Yes   
4         Oregon    M  Turquoise  Spring            2.7                 Yes   

   Shipping Type Discount Applied Promo Co

In [53]:
# ── HANDLE MISSING VALUES ─────────────────────────
# Number columns → fill with median
num_cols = df.select_dtypes(
    include=['float64', 'int64']
).columns

for col in num_cols:
    if df[col].isnull().sum() > 0:
        df[col] = df[col].fillna(df[col].median())
        print(f" Fixed {col} with median")

# Text columns → fill with Unknown
cat_cols = df.select_dtypes(include='object').columns
for col in cat_cols:
    if df[col].isnull().sum() > 0:
        df[col] = df[col].fillna('Unknown')
        print(f" Fixed {col} with Unknown")

# ── FIX TEXT CONSISTENCY ──────────────────────────
df['Gender']           = df['Gender'].str.strip().str.title()
df['Category']         = df['Category'].str.strip().str.title()
df['Location']         = df['Location'].str.strip().str.title()
df['Season']           = df['Season'].str.strip().str.title()
df['Payment Method']   = df['Payment Method'].str.strip().str.title()
df['Shipping Type']    = df['Shipping Type'].str.strip().str.title()
df['Discount Applied'] = df['Discount Applied'].str.strip().str.title()
df['Promo Code Used']  = df['Promo Code Used'].str.strip().str.title()
df['Subscription Status'] = df['Subscription Status'].str.strip().str.title()
df['Size']             = df['Size'].str.strip().str.upper()
df['Item Purchased']   = df['Item Purchased'].str.strip().str.title()
df['Color']            = df['Color'].str.strip().str.title()
df['Frequency of Purchases'] = df['Frequency of Purchases'].str.strip().str.title()

print()
print(" Text columns standardized!")

# ── REMOVE DUPLICATES ─────────────────────────────
before = len(df)
df     = df.drop_duplicates()
after  = len(df)
print(f"Duplicates removed: {before - after}")
print(f"   Rows remaining: {after}")

# ── VERIFY ────────────────────────────────────────
print()
print("Missing values after cleaning:")
print(df.isnull().sum().sum(), "total missing")

 Fixed Review Rating with median

 Text columns standardized!
Duplicates removed: 0
   Rows remaining: 3900

Missing values after cleaning:
0 total missing


In [54]:
# ── SAVE CLEANED FILE ─────────────────────────────
df.to_csv("cleaned_dataset.csv", index=False)

# ── VERIFY ────────────────────────────────────────
saved = pd.read_csv("cleaned_dataset.csv")
print(" cleaned_dataset.csv saved!")
print(f"   Rows    : {saved.shape[0]}")
print(f"   Columns : {saved.shape[1]}")
print(f"   Missing : {saved.isnull().sum().sum()}")

 cleaned_dataset.csv saved!
   Rows    : 3900
   Columns : 18
   Missing : 0


In [55]:
df = pd.read_csv("cleaned_dataset.csv")

print("=" * 55)
print("DATASET SUMMARY — SQL Retention Project")
print("=" * 55)

discount_pct = (df['Discount Applied'] == 'Yes').mean() * 100
promo_pct    = (df['Promo Code Used'] == 'Yes').mean() * 100

print(f"""
Total Customers      : {df['Customer ID'].nunique()}
Total Rows           : {len(df)}
Total Revenue        : ${df['Purchase Amount (USD)'].sum():,.2f}
Avg Purchase         : ${df['Purchase Amount (USD)'].mean():.2f}
Avg Review Rating    : {df['Review Rating'].mean():.2f} / 5
% Discount Used      : {discount_pct:.1f}%
% Promo Code Used    : {promo_pct:.1f}%
Top City             : {df['Location'].value_counts().index[0]}
Top Category         : {df['Category'].value_counts().index[0]}
Top Payment Method   : {df['Payment Method'].value_counts().index[0]}
Avg Prev Purchases   : {df['Previous Purchases'].mean():.1f}
""")

# ── YOUR KEY OBSERVATION ──────────────────────────
print("KEY OBSERVATIONS:")
print(f"""
1. PROMO DEPENDENCY:
   {discount_pct:.1f}% use discounts and {promo_pct:.1f}% use
   promo codes — same group uses both.
   Moderate dependency needs monitoring.

2. LOYALTY SIGNAL:
   Avg previous purchases = {df['Previous Purchases'].mean():.1f}
   Very high repeat buying — good loyalty sign!

3. REVENUE CONCERN:
   Avg purchase only ${df['Purchase Amount (USD)'].mean():.2f}
   Combined with {discount_pct:.1f}% promo use →
   discounts may be hurting revenue.
""")

# ── 5 BUSINESS QUESTIONS ──────────────────────────
print("5 BUSINESS QUESTIONS TO ANSWER:")
print("""
Q1: Who are genuinely loyal vs discount-driven?
Q2: What behavioral patterns predict high value?
Q3: Which cities/regions are underlevered?
Q4: How should brand restructure promo strategy?
Q5: What does ideal customer profile look like?
""")

print(" Ready for feature engineering!")

DATASET SUMMARY — SQL Retention Project

Total Customers      : 3900
Total Rows           : 3900
Total Revenue        : $233,081.00
Avg Purchase         : $59.76
Avg Review Rating    : 3.75 / 5
% Discount Used      : 43.0%
% Promo Code Used    : 43.0%
Top City             : Montana
Top Category         : Clothing
Top Payment Method   : Paypal
Avg Prev Purchases   : 25.4

KEY OBSERVATIONS:

1. PROMO DEPENDENCY:
   43.0% use discounts and 43.0% use
   promo codes — same group uses both.
   Moderate dependency needs monitoring.

2. LOYALTY SIGNAL:
   Avg previous purchases = 25.4
   Very high repeat buying — good loyalty sign!

3. REVENUE CONCERN:
   Avg purchase only $59.76
   Combined with 43.0% promo use →
   discounts may be hurting revenue.

5 BUSINESS QUESTIONS TO ANSWER:

Q1: Who are genuinely loyal vs discount-driven?
Q2: What behavioral patterns predict high value?
Q3: Which cities/regions are underlevered?
Q4: How should brand restructure promo strategy?
Q5: What does ideal cust

In [56]:
import pandas as pd
import numpy as np

# Load cleaned dataset
df = pd.read_csv("cleaned_dataset.csv")
print(f" Loaded: {df.shape}")

print("=" * 50)
print("DAY 8 — VALUE TIER FEATURE")
print("=" * 50)

# ── FEATURE 1A: Total Spent ───────────────────────
df['total_spent'] = df['Purchase Amount (USD)']

print("Spend distribution:")
print(df['total_spent'].describe().round(2))

# ── GET PERCENTILES ───────────────────────────────
p25 = df['total_spent'].quantile(0.25)
p50 = df['total_spent'].quantile(0.50)
p75 = df['total_spent'].quantile(0.75)

print(f"\nTier boundaries:")
print(f"  Bronze   → below ${p25:.2f}")
print(f"  Silver   → ${p25:.2f} to ${p50:.2f}")
print(f"  Gold     → ${p50:.2f} to ${p75:.2f}")
print(f"  Platinum → above ${p75:.2f}")

# ── FEATURE 1B: Value Tier ────────────────────────
df['value_tier'] = pd.cut(
    df['total_spent'],
    bins   = [0, p25, p50, p75, float('inf')],
    labels = ['Bronze', 'Silver', 'Gold', 'Platinum']
)

print("\nValue Tier distribution:")
print(df['value_tier'].value_counts())

# Save progress
df.to_csv("featured_dataset.csv", index=False)
print(" value_tier created!")

 Loaded: (3900, 18)
DAY 8 — VALUE TIER FEATURE
Spend distribution:
count    3900.00
mean       59.76
std        23.69
min        20.00
25%        39.00
50%        60.00
75%        81.00
max       100.00
Name: total_spent, dtype: float64

Tier boundaries:
  Bronze   → below $39.00
  Silver   → $39.00 to $60.00
  Gold     → $60.00 to $81.00
  Platinum → above $81.00

Value Tier distribution:
value_tier
Bronze      1014
Gold         988
Silver       972
Platinum     926
Name: count, dtype: int64
 value_tier created!


In [57]:
df = pd.read_csv("featured_dataset.csv")

print("=" * 50)
print("DAY 9 — PROMO DEPENDENCY SCORE")
print("=" * 50)

# ── FEATURE 2A: Individual Flags ──────────────────
df['discount_flag'] = (
    df['Discount Applied'] == 'Yes'
).astype(int)

df['promo_flag'] = (
    df['Promo Code Used'] == 'Yes'
).astype(int)

# ── FEATURE 2B: Promo Dependency Score ───────────
# 0.0 = used no promo
# 0.5 = used one promo
# 1.0 = used both discount + promo code
df['promo_dependency_score'] = (
    df['discount_flag'] + df['promo_flag']
) / 2

print("Promo Dependency Score:")
print(df['promo_dependency_score'].value_counts())
print()

# ── FEATURE 2C: Loyalty Type ──────────────────────
df['loyalty_type'] = df['promo_dependency_score'].apply(
    lambda x:
    'Genuinely Loyal'    if x == 0.0 else
    'Moderate'           if x == 0.5 else
    'Discount Dependent'
)

print("Loyalty Type distribution:")
print(df['loyalty_type'].value_counts())
print()
print("Avg spend by loyalty type:")
print(df.groupby('loyalty_type')[
    'total_spent'
].mean().round(2))

df.to_csv("featured_dataset.csv", index=False)
print(" promo_dependency_score created!")

DAY 9 — PROMO DEPENDENCY SCORE
Promo Dependency Score:
promo_dependency_score
0.0    2223
1.0    1677
Name: count, dtype: int64

Loyalty Type distribution:
loyalty_type
Genuinely Loyal       2223
Discount Dependent    1677
Name: count, dtype: int64

Avg spend by loyalty type:
loyalty_type
Discount Dependent    59.28
Genuinely Loyal       60.13
Name: total_spent, dtype: float64
 promo_dependency_score created!


In [58]:
df = pd.read_csv("featured_dataset.csv")

print("=" * 50)
print("DAY 10 — SATISFACTION + REPEAT BUYER")
print("=" * 50)

# ── FEATURE 3A: Satisfaction Flag ────────────────
avg_rating = df['Review Rating'].mean()
print(f"Average rating: {avg_rating:.2f}")

df['satisfaction_flag'] = df['Review Rating'].apply(
    lambda x: 'High' if x >= avg_rating else 'Low'
)

print("\nSatisfaction Flag:")
print(df['satisfaction_flag'].value_counts())
print()
print("Avg spend by satisfaction:")
print(df.groupby('satisfaction_flag')[
    'total_spent'
].mean().round(2))

# ── FEATURE 3B: Repeat Buyer Segment ─────────────
pp25 = df['Previous Purchases'].quantile(0.25)
pp75 = df['Previous Purchases'].quantile(0.75)

print(f"\nRepeat buyer boundaries:")
print(f"  Low    → below {pp25:.0f} purchases")
print(f"  Medium → {pp25:.0f} to {pp75:.0f} purchases")
print(f"  High   → above {pp75:.0f} purchases")

df['repeat_buyer_segment'] = pd.cut(
    df['Previous Purchases'],
    bins   = [0, pp25, pp75, float('inf')],
    labels = ['Low', 'Medium', 'High']
)

print("\nRepeat Buyer Segment:")
print(df['repeat_buyer_segment'].value_counts())

# ── FEATURE 3C: Subscriber Flag ───────────────────
df['is_subscriber'] = (
    df['Subscription Status'] == 'Yes'
).astype(int)

print("\nSubscriber distribution:")
print(df['Subscription Status'].value_counts())

df.to_csv("featured_dataset.csv", index=False)
print(" Satisfaction + Repeat features done!")

DAY 10 — SATISFACTION + REPEAT BUYER
Average rating: 3.75

Satisfaction Flag:
satisfaction_flag
High    1974
Low     1926
Name: count, dtype: int64

Avg spend by satisfaction:
satisfaction_flag
High    60.51
Low     59.00
Name: total_spent, dtype: float64

Repeat buyer boundaries:
  Low    → below 13 purchases
  Medium → 13 to 38 purchases
  High   → above 38 purchases

Repeat Buyer Segment:
repeat_buyer_segment
Medium    1958
Low       1014
High       928
Name: count, dtype: int64

Subscriber distribution:
Subscription Status
No     2847
Yes    1053
Name: count, dtype: int64
 Satisfaction + Repeat features done!


In [59]:
df = pd.read_csv("featured_dataset.csv")

print("=" * 50)
print("DAY 11 — CUSTOMER MASTER TABLE")
print("=" * 50)

# ── SELECT ALL COLUMNS ────────────────────────────
customer_master = df[[
    'Customer ID', 'Age', 'Gender', 'Location',
    'total_spent', 'Purchase Amount (USD)',
    'Previous Purchases', 'Frequency of Purchases',
    'Payment Method', 'Category', 'Item Purchased',
    'Season', 'Size', 'Color', 'Shipping Type',
    'value_tier', 'promo_dependency_score',
    'loyalty_type', 'satisfaction_flag',
    'repeat_buyer_segment', 'is_subscriber',
    'discount_flag', 'promo_flag',
    'Discount Applied', 'Promo Code Used',
    'Review Rating', 'Subscription Status',
]].copy()

print(f"Master table shape: {customer_master.shape}")

# ── VALIDATE FEATURES ─────────────────────────────
print("\nVALIDATION:")

print("\nAvg spend by value tier:")
print(customer_master.groupby(
    'value_tier', observed=True
)['total_spent'].mean().round(2))

print("\nAvg promo score by loyalty:")
print(customer_master.groupby(
    'loyalty_type'
)['promo_dependency_score'].mean().round(2))

print("\nAvg rating by satisfaction:")
print(customer_master.groupby(
    'satisfaction_flag'
)['Review Rating'].mean().round(2))

# Save
customer_master.to_csv(
    "customer_master.csv", index=False
)
print(" customer_master.csv saved!")

DAY 11 — CUSTOMER MASTER TABLE
Master table shape: (3900, 27)

VALIDATION:

Avg spend by value tier:
value_tier
Bronze      29.52
Gold        71.05
Platinum    91.09
Silver      50.00
Name: total_spent, dtype: float64

Avg promo score by loyalty:
loyalty_type
Discount Dependent    1.0
Genuinely Loyal       0.0
Name: promo_dependency_score, dtype: float64

Avg rating by satisfaction:
satisfaction_flag
High    4.36
Low     3.13
Name: Review Rating, dtype: float64
 customer_master.csv saved!


In [60]:
df = pd.read_csv("customer_master.csv")

print("=" * 50)
print("DAY 12 — GEOGRAPHIC + CATEGORY FEATURES")
print("=" * 50)

# ── CITY STATS ────────────────────────────────────
city_stats = df.groupby('Location').agg(
    city_avg_spend    = ('total_spent', 'mean'),
    city_promo_pct    = ('promo_dependency_score', 'mean'),
    city_customer_count = ('Customer ID', 'count'),
    city_avg_rating   = ('Review Rating', 'mean')
).reset_index().round(2)

# City opportunity labels
avg_city_spend = city_stats['city_avg_spend'].mean()
avg_city_promo = city_stats['city_promo_pct'].mean()

city_stats['city_opportunity'] = city_stats.apply(
    lambda row:
    'Golden Market'  if row['city_avg_spend'] > avg_city_spend
                     and row['city_promo_pct'] < avg_city_promo
    else 'Promo Market'   if row['city_avg_spend'] > avg_city_spend
                     and row['city_promo_pct'] >= avg_city_promo
    else 'Growth Market'  if row['city_avg_spend'] <= avg_city_spend
                     and row['city_promo_pct'] < avg_city_promo
    else 'Low Priority',
    axis=1
)

print("City opportunity breakdown:")
print(city_stats['city_opportunity'].value_counts())

# Merge back
df = df.merge(
    city_stats[[
        'Location', 'city_opportunity',
        'city_avg_spend', 'city_promo_pct'
    ]],
    on='Location', how='left'
)

# ── CATEGORY STATS ────────────────────────────────
cat_stats = df.groupby('Category').agg(
    cat_avg_spend      = ('total_spent', 'mean'),
    cat_promo_pct      = ('promo_dependency_score', 'mean'),
    cat_avg_prev_purch = ('Previous Purchases', 'mean'),
).reset_index().round(2)

avg_prev = cat_stats['cat_avg_prev_purch'].mean()
cat_stats['category_type'] = cat_stats[
    'cat_avg_prev_purch'
].apply(
    lambda x: 'Retention Category'
    if x >= avg_prev else 'Entry Category'
)

print("\nCategory types:")
print(cat_stats[['Category',
                 'cat_avg_prev_purch',
                 'category_type']])

df = df.merge(
    cat_stats[[
        'Category', 'category_type',
        'cat_avg_spend', 'cat_promo_pct'
    ]],
    on='Category', how='left'
)

df.to_csv("customer_master.csv", index=False)
print(" Geographic + category features done!")

DAY 12 — GEOGRAPHIC + CATEGORY FEATURES
City opportunity breakdown:
city_opportunity
Low Priority     14
Golden Market    13
Promo Market     13
Growth Market    10
Name: count, dtype: int64

Category types:
      Category  cat_avg_prev_purch       category_type
0  Accessories               25.73  Retention Category
1     Clothing               25.20      Entry Category
2     Footwear               25.23      Entry Category
3    Outerwear               24.96      Entry Category
 Geographic + category features done!


In [61]:
df = pd.read_csv("customer_master.csv")

print("=" * 55)
print("DAY 13 — FINAL FEATURE DOCUMENTATION")
print("=" * 55)

print(f"\nTotal columns    : {len(df.columns)}")
print(f"Total customers  : {len(df)}")

print("\nFEATURE DOCUMENTATION:")
print("-" * 55)

features = {
    'value_tier'            : 'Spending tier based on purchase amount percentiles — Bronze/Silver/Gold/Platinum',
    'promo_dependency_score': '0=no promo used, 0.5=one promo, 1.0=both discount+promo code used',
    'loyalty_type'          : 'Genuinely Loyal / Moderate / Discount Dependent based on promo score',
    'satisfaction_flag'     : 'High if rating >= dataset average, Low otherwise',
    'repeat_buyer_segment'  : 'Low/Medium/High based on Previous Purchases percentiles',
    'is_subscriber'         : '1=subscribed, 0=not subscribed',
    'city_opportunity'      : 'Golden/Promo/Growth/Low Priority based on spend vs promo dependency',
    'category_type'         : 'Entry Category or Retention Category based on avg previous purchases',
}

for feat, desc in features.items():
    print(f"\n * {feat}")
    print(f"   {desc}")

# Final validation
print("\n" + "=" * 55)
print("FINAL VALIDATION")
print("=" * 55)
print(f"Missing values : {df.isnull().sum().sum()}")
print(f"Duplicates     : {df.duplicated().sum()}")

print("\nValue Tier counts:")
print(df['value_tier'].value_counts())
print("\nLoyalty Type counts:")
print(df['loyalty_type'].value_counts())
print("\nCity Opportunity counts:")
print(df['city_opportunity'].value_counts())
print("\nCategory Type counts:")
print(df['category_type'].value_counts())

# Save final
df.to_csv("customer_master_final.csv", index=False)
print("\n customer_master_final.csv saved!")

DAY 13 — FINAL FEATURE DOCUMENTATION

Total columns    : 33
Total customers  : 3900

FEATURE DOCUMENTATION:
-------------------------------------------------------

 * value_tier
   Spending tier based on purchase amount percentiles — Bronze/Silver/Gold/Platinum

 * promo_dependency_score
   0=no promo used, 0.5=one promo, 1.0=both discount+promo code used

 * loyalty_type
   Genuinely Loyal / Moderate / Discount Dependent based on promo score

 * satisfaction_flag
   High if rating >= dataset average, Low otherwise

 * repeat_buyer_segment
   Low/Medium/High based on Previous Purchases percentiles

 * is_subscriber
   1=subscribed, 0=not subscribed

 * city_opportunity
   Golden/Promo/Growth/Low Priority based on spend vs promo dependency

 * category_type
   Entry Category or Retention Category based on avg previous purchases

FINAL VALIDATION
Missing values : 0
Duplicates     : 0

Value Tier counts:
value_tier
Bronze      1014
Gold         988
Silver       972
Platinum     926
Name:

In [62]:
df = pd.read_csv("customer_master_final.csv")

print("=" * 55)
print("DAY 14 — WEEK 2 REVISION")
print("=" * 55)

# Answer all key questions
loyal    = df[df['loyalty_type'] == 'Genuinely Loyal']
platinum = df[df['value_tier']   == 'Platinum']
golden   = df[df['city_opportunity'] == 'Golden Market']
high_sat = (df['satisfaction_flag'] == 'High').mean() * 100
ret_cats = df[
    df['category_type'] == 'Retention Category'
]['Category'].unique()

print(f"\n1. Genuinely Loyal customers : {len(loyal)} ({len(loyal)/len(df)*100:.1f}%)")
print(f"2. Platinum customers        : {len(platinum)} ({len(platinum)/len(df)*100:.1f}%)")
print(f"3. Golden Market customers   : {len(golden)}")
print(f"4. Retention categories      : {ret_cats}")
print(f"5. High satisfaction %       : {high_sat:.1f}%")

print(f"""

Files saved:
→ cleaned_dataset.csv
→ featured_dataset.csv
→ customer_master.csv
→ customer_master_final.csv

Features created:
→ value_tier
→ promo_dependency_score
→ loyalty_type
→ satisfaction_flag
→ repeat_buyer_segment
→ is_subscriber
→ city_opportunity
→ category_type

""")

DAY 14 — WEEK 2 REVISION

1. Genuinely Loyal customers : 2223 (57.0%)
2. Platinum customers        : 926 (23.7%)
3. Golden Market customers   : 1013
4. Retention categories      : ['Accessories']
5. High satisfaction %       : 50.6%


Files saved:
→ cleaned_dataset.csv
→ featured_dataset.csv
→ customer_master.csv
→ customer_master_final.csv

Features created:
→ value_tier
→ promo_dependency_score
→ loyalty_type
→ satisfaction_flag
→ repeat_buyer_segment
→ is_subscriber
→ city_opportunity
→ category_type




In [63]:
import pandas as pd
import mysql.connector
from sqlalchemy import create_engine
import os

# ── YOUR MYSQL CREDENTIALS ────────────────────────
HOST     = "localhost"
USER     = "root"
PASSWORD = "1234"  
DATABASE = "sql_retention_db"

# ── STEP 1: CREATE DATABASE ───────────────────────
print("Creating database...")
setup_conn = mysql.connector.connect(
    host     = HOST,
    user     = USER,
    password = PASSWORD
)
cursor = setup_conn.cursor()
cursor.execute(
    f"CREATE DATABASE IF NOT EXISTS {DATABASE}"
)
print(f" Database '{DATABASE}' created!")
cursor.close()
setup_conn.close()

# ── STEP 2: CREATE ENGINE ─────────────────────────
engine = create_engine(
    f"mysql+pymysql://{USER}:{PASSWORD}@{HOST}/{DATABASE}"
)
print(" Engine created!")

# ── STEP 3: LOAD + CLEAN COLUMN NAMES ────────────
df = pd.read_csv("customer_master_final.csv")

# Clean column names for MySQL
df.columns = (
    df.columns
    .str.strip()
    .str.replace(' ', '_')
    .str.replace('(', '')
    .str.replace(')', '')
    .str.lower()
)

print("\nCleaned column names:")
for col in df.columns:
    print(f"  → {col}")

# ── STEP 4: PUSH TO MYSQL ─────────────────────────
df.to_sql(
    name      = "customer_master",
    con       = engine,
    if_exists = "replace",
    index     = False,
    chunksize = 1000
)

print(f"\n✅ Table loaded!")
print(f"   Rows : {len(df):,}")
print(f"   Cols : {len(df.columns)}")

# ── STEP 5: HELPER FUNCTION ───────────────────────
def run_sql(query):
    result = pd.read_sql_query(query, engine)
    print(result.to_string(index=False))
    print(f"\nRows: {len(result)}")
    return result

# ── STEP 6: VERIFY ────────────────────────────────
print("\nVerifying...")
run_sql("SELECT COUNT(*) AS total_rows FROM customer_master")
run_sql("SHOW COLUMNS FROM customer_master")

Creating database...
 Database 'sql_retention_db' created!
 Engine created!

Cleaned column names:
  → customer_id
  → age
  → gender
  → location
  → total_spent
  → purchase_amount_usd
  → previous_purchases
  → frequency_of_purchases
  → payment_method
  → category
  → item_purchased
  → season
  → size
  → color
  → shipping_type
  → value_tier
  → promo_dependency_score
  → loyalty_type
  → satisfaction_flag
  → repeat_buyer_segment
  → is_subscriber
  → discount_flag
  → promo_flag
  → discount_applied
  → promo_code_used
  → review_rating
  → subscription_status
  → city_opportunity
  → city_avg_spend
  → city_promo_pct
  → category_type
  → cat_avg_spend
  → cat_promo_pct

✅ Table loaded!
   Rows : 3,900
   Cols : 33

Verifying...
 total_rows
       3900

Rows: 1
                 Field   Type Null Key Default Extra
           customer_id bigint  YES        None      
                   age bigint  YES        None      
                gender   text  YES        None      
      

,Field,Type,Null,Key,Default,Extra
0,customer_id,bigint,YES,,None,
1,age,bigint,YES,,None,
2,gender,text,YES,,None,
3,location,text,YES,,None,
4,total_spent,bigint,YES,,None,
5,purchase_amount_usd,bigint,YES,,None,
6,previous_purchases,bigint,YES,,None,
7,frequency_of_purchases,text,YES,,None,
8,payment_method,text,YES,,None,
9,category,text,YES,,None,


In [64]:
print("=" * 60)
print("Q1 — LOYAL vs DISCOUNT CUSTOMERS")
print("=" * 60)

# ── 1A: Overall Loyalty Breakdown ─────────────────
print("\n 1A: Overall Loyalty Breakdown")
run_sql("""
    SELECT
        loyalty_type,
        COUNT(*)                           AS customers,
        ROUND(COUNT(*) * 100.0 / 3900, 1) AS pct_of_total,
        ROUND(AVG(total_spent), 2)         AS avg_spend,
        ROUND(AVG(review_rating), 2)       AS avg_rating,
        ROUND(AVG(previous_purchases), 1)  AS avg_prev_purchases
    FROM customer_master
    GROUP BY loyalty_type
    ORDER BY avg_spend DESC
""")

# ── 1B: Loyalty vs Value Tier ─────────────────────
print("\n 1B: Loyalty Type vs Value Tier")
run_sql("""
    SELECT
        loyalty_type,
        value_tier,
        COUNT(*)                   AS customers,
        ROUND(AVG(total_spent), 2) AS avg_spend
    FROM customer_master
    GROUP BY loyalty_type, value_tier
    ORDER BY loyalty_type, avg_spend DESC
""")

# ── 1C: Loyalty vs Subscription ───────────────────
print("\n1C: Loyalty vs Subscription")
run_sql("""
    SELECT
        loyalty_type,
        subscription_status,
        COUNT(*)                          AS customers,
        ROUND(AVG(total_spent), 2)        AS avg_spend,
        ROUND(AVG(previous_purchases), 1) AS avg_purchases
    FROM customer_master
    GROUP BY loyalty_type, subscription_status
    ORDER BY loyalty_type, avg_spend DESC
""")

# ── 1D: Champion Customers ────────────────────────
print("\n 1D: True Champion Customers")
print("(Genuinely Loyal + Platinum + High Satisfaction)")
run_sql("""
    SELECT
        COUNT(*)                           AS champion_customers,
        ROUND(COUNT(*) * 100.0 / 3900, 1) AS pct_of_total,
        ROUND(AVG(total_spent), 2)         AS avg_spend,
        ROUND(AVG(previous_purchases), 1)  AS avg_purchases,
        ROUND(AVG(review_rating), 2)       AS avg_rating
    FROM customer_master
    WHERE loyalty_type      = 'Genuinely Loyal'
      AND value_tier        = 'Platinum'
      AND satisfaction_flag = 'High'
""")

print("Q1 fully answered!")

Q1 — LOYAL vs DISCOUNT CUSTOMERS

 1A: Overall Loyalty Breakdown
      loyalty_type  customers  pct_of_total  avg_spend  avg_rating  avg_prev_purchases
   Genuinely Loyal       2223          57.0      60.13        3.76                25.1
Discount Dependent       1677          43.0      59.28        3.74                25.7

Rows: 2

 1B: Loyalty Type vs Value Tier
      loyalty_type value_tier  customers  avg_spend
Discount Dependent   Platinum        381      91.00
Discount Dependent       Gold        438      70.94
Discount Dependent     Silver        412      49.89
Discount Dependent     Bronze        446      29.40
   Genuinely Loyal   Platinum        545      91.15
   Genuinely Loyal       Gold        550      71.14
   Genuinely Loyal     Silver        560      50.08
   Genuinely Loyal     Bronze        568      29.61

Rows: 8

1C: Loyalty vs Subscription
      loyalty_type subscription_status  customers  avg_spend  avg_purchases
Discount Dependent                 Yes       1053 

In [65]:
print("=" * 60)
print("Q2 — BEHAVIORAL PATTERNS PREDICTING HIGH VALUE")
print("=" * 60)

# ── 2A: Value Tier Profile ────────────────────────
print("\n 2A: Behavioral Profile by Value Tier")
run_sql("""
    SELECT
        value_tier,
        COUNT(*)                              AS customers,
        ROUND(AVG(total_spent), 2)            AS avg_spend,
        ROUND(AVG(previous_purchases), 1)     AS avg_prev_purchases,
        ROUND(AVG(review_rating), 2)          AS avg_rating,
        ROUND(AVG(promo_dependency_score), 2) AS avg_promo_dependency,
        ROUND(AVG(is_subscriber), 2)          AS subscriber_rate
    FROM customer_master
    GROUP BY value_tier
    ORDER BY avg_spend DESC
""")

# ── 2B: Purchase Frequency ────────────────────────
print("\n 2B: Purchase Frequency vs Value")
run_sql("""
    SELECT
        frequency_of_purchases,
        COUNT(*)                              AS customers,
        ROUND(AVG(total_spent), 2)            AS avg_spend,
        ROUND(AVG(previous_purchases), 1)     AS avg_prev_purchases,
        ROUND(AVG(promo_dependency_score), 2) AS avg_promo_dependency
    FROM customer_master
    GROUP BY frequency_of_purchases
    ORDER BY avg_prev_purchases DESC
""")

# ── 2C: Payment Method ────────────────────────────
print("\n 2C: Payment Method vs Value")
run_sql("""
    SELECT
        payment_method,
        COUNT(*)                              AS customers,
        ROUND(AVG(total_spent), 2)            AS avg_spend,
        ROUND(AVG(promo_dependency_score), 2) AS avg_promo_dependency,
        ROUND(AVG(previous_purchases), 1)     AS avg_prev_purchases
    FROM customer_master
    GROUP BY payment_method
    ORDER BY avg_spend DESC
""")

# ── 2D: Age Group ─────────────────────────────────
print("\n 2D: Age Group vs Value")
run_sql("""
    SELECT
        CASE
            WHEN age BETWEEN 18 AND 25 THEN '18-25'
            WHEN age BETWEEN 26 AND 35 THEN '26-35'
            WHEN age BETWEEN 36 AND 45 THEN '36-45'
            WHEN age BETWEEN 46 AND 55 THEN '46-55'
            ELSE '56+'
        END                                   AS age_group,
        COUNT(*)                              AS customers,
        ROUND(AVG(total_spent), 2)            AS avg_spend,
        ROUND(AVG(previous_purchases), 1)     AS avg_prev_purchases,
        ROUND(AVG(promo_dependency_score), 2) AS avg_promo_dependency
    FROM customer_master
    GROUP BY age_group
    ORDER BY avg_spend DESC
""")

print(" Q2 fully answered!")

Q2 — BEHAVIORAL PATTERNS PREDICTING HIGH VALUE

 2A: Behavioral Profile by Value Tier
value_tier  customers  avg_spend  avg_prev_purchases  avg_rating  avg_promo_dependency  subscriber_rate
  Platinum        926      91.09                25.5        3.78                  0.41             0.26
      Gold        988      71.05                25.6        3.79                  0.44             0.27
    Silver        972      50.00                25.2        3.67                  0.42             0.27
    Bronze       1014      29.52                25.2        3.76                  0.44             0.27

Rows: 4

 2B: Purchase Frequency vs Value
frequency_of_purchases  customers  avg_spend  avg_prev_purchases  avg_promo_dependency
             Quarterly        563      59.98                26.9                  0.43
                Weekly        539      58.97                25.8                  0.43
           Fortnightly        542      59.05                25.3                  0.45
   

In [66]:
print("=" * 60)
print("Q3 — GEOGRAPHIC OPPORTUNITY ANALYSIS")
print("=" * 60)

# ── 3A: City Opportunity Summary ──────────────────
print("\n 3A: City Opportunity Breakdown")
run_sql("""
    SELECT
        city_opportunity,
        COUNT(DISTINCT location)              AS num_cities,
        COUNT(*)                              AS total_customers,
        ROUND(AVG(total_spent), 2)            AS avg_spend,
        ROUND(AVG(promo_dependency_score), 2) AS avg_promo_dependency,
        ROUND(AVG(previous_purchases), 1)     AS avg_prev_purchases
    FROM customer_master
    GROUP BY city_opportunity
    ORDER BY avg_spend DESC
""")

# ── 3B: Top Golden Market Cities ──────────────────
print("\n 3B: Top Golden Market Cities")
run_sql("""
    SELECT
        location,
        COUNT(*)                              AS customers,
        ROUND(AVG(total_spent), 2)            AS avg_spend,
        ROUND(AVG(promo_dependency_score), 2) AS promo_dependency,
        ROUND(AVG(previous_purchases), 1)     AS avg_prev_purchases
    FROM customer_master
    WHERE city_opportunity = 'Golden Market'
    GROUP BY location
    ORDER BY avg_spend DESC
    LIMIT 10
""")

# ── 3C: Promo Market Warning ──────────────────────
print("\n 3C: Promo Market Cities")
run_sql("""
    SELECT
        location,
        COUNT(*)                              AS customers,
        ROUND(AVG(total_spent), 2)            AS avg_spend,
        ROUND(AVG(promo_dependency_score), 2) AS promo_dependency,
        ROUND(SUM(total_spent), 2)            AS total_revenue
    FROM customer_master
    WHERE city_opportunity = 'Promo Market'
    GROUP BY location
    ORDER BY total_revenue DESC
    LIMIT 10
""")

# ── 3D: Category by City Opportunity ─────────────
print("\n 3D: What Do Golden Cities Buy?")
run_sql("""
    SELECT
        city_opportunity,
        category,
        COUNT(*)                   AS customers,
        ROUND(AVG(total_spent), 2) AS avg_spend
    FROM customer_master
    WHERE city_opportunity IN (
        'Golden Market', 'Growth Market'
    )
    GROUP BY city_opportunity, category
    ORDER BY city_opportunity, customers DESC
""")

print(" Q3 geographic analysis done!")

Q3 — GEOGRAPHIC OPPORTUNITY ANALYSIS

 3A: City Opportunity Breakdown
city_opportunity  num_cities  total_customers  avg_spend  avg_promo_dependency  avg_prev_purchases
    Promo Market          13             1005      62.29                  0.47                25.5
   Golden Market          13             1013      61.91                  0.39                25.4
    Low Priority          14             1090      57.31                  0.47                25.8
   Growth Market          10              792      57.19                  0.38                24.4

Rows: 4

 3B: Top Golden Market Cities
    location  customers  avg_spend  promo_dependency  avg_prev_purchases
      Alaska         72      67.60              0.40                28.1
     Arizona         65      66.55              0.34                28.4
    Virginia         77      62.88              0.38                23.5
    Michigan         73      62.10              0.40                26.9
   Tennessee         77      6

In [67]:
print("=" * 60)
print("Q4 — PROMOTIONAL RESTRUCTURING STRATEGY")
print("=" * 60)

# ── 4A: Two Loyalty Definitions ───────────────────
print("\n 4A: Loyalty Definition 1 — Promo Dependency")
run_sql("""
    SELECT
        loyalty_type,
        COUNT(*)                          AS customers,
        ROUND(AVG(total_spent), 2)        AS avg_spend,
        ROUND(AVG(previous_purchases), 1) AS avg_purchases,
        ROUND(AVG(review_rating), 2)      AS avg_rating
    FROM customer_master
    GROUP BY loyalty_type
    ORDER BY avg_spend DESC
""")

print("\n Loyalty Definition 2 — Repeat Purchase Segment")
run_sql("""
    SELECT
        repeat_buyer_segment,
        COUNT(*)                              AS customers,
        ROUND(AVG(total_spent), 2)            AS avg_spend,
        ROUND(AVG(promo_dependency_score), 2) AS avg_promo_dependency,
        ROUND(AVG(review_rating), 2)          AS avg_rating
    FROM customer_master
    GROUP BY repeat_buyer_segment
    ORDER BY avg_spend DESC
""")

# ── 4B: Promo Sunset Plan ─────────────────────────
print("\n 4B: Promo Sunset Plan")
run_sql("""
    SELECT
        loyalty_type,
        value_tier,
        COUNT(*)                              AS customers,
        ROUND(AVG(total_spent), 2)            AS avg_spend,
        ROUND(AVG(promo_dependency_score), 2) AS promo_score,
        CASE
            WHEN loyalty_type = 'Genuinely Loyal'
             AND value_tier IN ('Gold', 'Platinum')
            THEN 'STOP Discounts — Safe'
            WHEN loyalty_type = 'Genuinely Loyal'
             AND value_tier IN ('Silver', 'Bronze')
            THEN 'REDUCE Discounts Gradually'
            WHEN loyalty_type = 'Discount Dependent'
             AND value_tier IN ('Gold', 'Platinum')
            THEN 'KEEP Discounts — High Risk'
            ELSE 'LOW PRIORITY — Monitor Only'
        END AS promo_action
    FROM customer_master
    GROUP BY loyalty_type, value_tier
    ORDER BY avg_spend DESC
""")

# ── 4C: Revenue Impact ────────────────────────────
print("\n 4C: Potential Margin Recovery")
run_sql("""
    SELECT
        loyalty_type,
        COUNT(*)                           AS customers,
        ROUND(SUM(total_spent), 2)         AS current_revenue,
        ROUND(AVG(total_spent), 2)         AS avg_spend,
        ROUND(SUM(total_spent) * 0.15, 2) AS potential_margin_gain
    FROM customer_master
    WHERE loyalty_type = 'Genuinely Loyal'
    GROUP BY loyalty_type
""")

print(" Q4 promo strategy answered!")

Q4 — PROMOTIONAL RESTRUCTURING STRATEGY

 4A: Loyalty Definition 1 — Promo Dependency
      loyalty_type  customers  avg_spend  avg_purchases  avg_rating
   Genuinely Loyal       2223      60.13           25.1        3.76
Discount Dependent       1677      59.28           25.7        3.74

Rows: 2

 Loyalty Definition 2 — Repeat Purchase Segment
repeat_buyer_segment  customers  avg_spend  avg_promo_dependency  avg_rating
                 Low       1014      60.14                  0.41        3.76
                High        928      59.98                  0.45        3.77
              Medium       1958      59.47                  0.43        3.74

Rows: 3

 4B: Promo Sunset Plan
      loyalty_type value_tier  customers  avg_spend  promo_score                promo_action
   Genuinely Loyal   Platinum        545      91.15          0.0       STOP Discounts — Safe
Discount Dependent   Platinum        381      91.00          1.0  KEEP Discounts — High Risk
   Genuinely Loyal       Gold   

In [68]:
# ── 5C: Fixed ICP Summary ─────────────────────────
print("\n 5C: IDEAL CUSTOMER PROFILE SUMMARY")

run_sql("""
    SELECT 'Top Gender' AS attribute,
            gender      AS value,
            COUNT(*)    AS count_val
    FROM customer_master
    WHERE loyalty_type      = 'Genuinely Loyal'
      AND value_tier        = 'Platinum'
      AND satisfaction_flag = 'High'
    GROUP BY gender
    HAVING COUNT(*) = (
        SELECT MAX(cnt) FROM (
            SELECT COUNT(*) AS cnt
            FROM customer_master
            WHERE loyalty_type      = 'Genuinely Loyal'
              AND value_tier        = 'Platinum'
              AND satisfaction_flag = 'High'
            GROUP BY gender
        ) t
    )

    UNION ALL

    SELECT 'Top Payment', payment_method, COUNT(*)
    FROM customer_master
    WHERE loyalty_type      = 'Genuinely Loyal'
      AND value_tier        = 'Platinum'
      AND satisfaction_flag = 'High'
    GROUP BY payment_method
    HAVING COUNT(*) = (
        SELECT MAX(cnt) FROM (
            SELECT COUNT(*) AS cnt
            FROM customer_master
            WHERE loyalty_type      = 'Genuinely Loyal'
              AND value_tier        = 'Platinum'
              AND satisfaction_flag = 'High'
            GROUP BY payment_method
        ) t
    )

    UNION ALL

    SELECT 'Top Category', category, COUNT(*)
    FROM customer_master
    WHERE loyalty_type      = 'Genuinely Loyal'
      AND value_tier        = 'Platinum'
      AND satisfaction_flag = 'High'
    GROUP BY category
    HAVING COUNT(*) = (
        SELECT MAX(cnt) FROM (
            SELECT COUNT(*) AS cnt
            FROM customer_master
            WHERE loyalty_type      = 'Genuinely Loyal'
              AND value_tier        = 'Platinum'
              AND satisfaction_flag = 'High'
            GROUP BY category
        ) t
    )

    UNION ALL

    SELECT 'Top Frequency', frequency_of_purchases, COUNT(*)
    FROM customer_master
    WHERE loyalty_type      = 'Genuinely Loyal'
      AND value_tier        = 'Platinum'
      AND satisfaction_flag = 'High'
    GROUP BY frequency_of_purchases
    HAVING COUNT(*) = (
        SELECT MAX(cnt) FROM (
            SELECT COUNT(*) AS cnt
            FROM customer_master
            WHERE loyalty_type      = 'Genuinely Loyal'
              AND value_tier        = 'Platinum'
              AND satisfaction_flag = 'High'
            GROUP BY frequency_of_purchases
        ) t
    )

    UNION ALL

    SELECT 'Top Location', location, COUNT(*)
    FROM customer_master
    WHERE loyalty_type      = 'Genuinely Loyal'
      AND value_tier        = 'Platinum'
      AND satisfaction_flag = 'High'
    GROUP BY location
    HAVING COUNT(*) = (
        SELECT MAX(cnt) FROM (
            SELECT COUNT(*) AS cnt
            FROM customer_master
            WHERE loyalty_type      = 'Genuinely Loyal'
              AND value_tier        = 'Platinum'
              AND satisfaction_flag = 'High'
            GROUP BY location
        ) t
    )
""")

print(" Q5 Ideal Customer Profile done!")
print("All 5 business questions answered! 🎉")


 5C: IDEAL CUSTOMER PROFILE SUMMARY
    attribute    value  count_val
   Top Gender   Female        149
  Top Payment   Paypal         59
 Top Category Clothing        132
Top Frequency Annually         57
 Top Location    Texas         12

Rows: 5
 Q5 Ideal Customer Profile done!
All 5 business questions answered! 🎉


In [70]:
import pandas as pd
from sqlalchemy import create_engine

# Reconnect if needed
HOST     = "localhost"
USER     = "root"
PASSWORD = "your_password"  # ← your password
DATABASE = "sql_retention_db"

engine = create_engine(
    f"mysql+pymysql://{USER}:{1234}@{HOST}/{DATABASE}"
)

print("Saving all results for Power BI...")
print("=" * 50)

# ── SAVE 1: Loyalty Summary ───────────────────────
loyalty_summary = pd.read_sql_query("""
    SELECT
        loyalty_type,
        value_tier,
        COUNT(*)                              AS customers,
        ROUND(AVG(total_spent), 2)            AS avg_spend,
        ROUND(AVG(promo_dependency_score), 2) AS avg_promo_dependency,
        ROUND(AVG(previous_purchases), 1)     AS avg_purchases,
        ROUND(AVG(review_rating), 2)          AS avg_rating
    FROM customer_master
    GROUP BY loyalty_type, value_tier
    ORDER BY avg_spend DESC
""", engine)

loyalty_summary.to_csv(
    "powerbi_loyalty_summary.csv", index=False
)
print(f" powerbi_loyalty_summary.csv → {len(loyalty_summary)} rows")

# ── SAVE 2: Geographic Summary ────────────────────
geo_summary = pd.read_sql_query("""
    SELECT
        location,
        city_opportunity,
        COUNT(*)                              AS customers,
        ROUND(AVG(total_spent), 2)            AS avg_spend,
        ROUND(AVG(promo_dependency_score), 2) AS promo_dependency,
        ROUND(SUM(total_spent), 2)            AS total_revenue
    FROM customer_master
    GROUP BY location, city_opportunity
    ORDER BY avg_spend DESC
""", engine)

geo_summary.to_csv(
    "powerbi_geo_summary.csv", index=False
)
print(f" powerbi_geo_summary.csv → {len(geo_summary)} rows")

# ── SAVE 3: Category Summary ──────────────────────
cat_summary = pd.read_sql_query("""
    SELECT
        category,
        category_type,
        COUNT(*)                              AS customers,
        ROUND(AVG(total_spent), 2)            AS avg_spend,
        ROUND(AVG(previous_purchases), 1)     AS avg_prev_purchases,
        ROUND(AVG(promo_dependency_score), 2) AS promo_dependency
    FROM customer_master
    GROUP BY category, category_type
    ORDER BY avg_prev_purchases DESC
""", engine)

cat_summary.to_csv(
    "powerbi_category_summary.csv", index=False
)
print(f" powerbi_category_summary.csv → {len(cat_summary)} rows")

# ── SAVE 4: Promo Sunset Plan ─────────────────────
promo_summary = pd.read_sql_query("""
    SELECT
        loyalty_type,
        value_tier,
        COUNT(*)                              AS customers,
        ROUND(AVG(total_spent), 2)            AS avg_spend,
        ROUND(AVG(promo_dependency_score), 2) AS promo_score,
        CASE
            WHEN loyalty_type = 'Genuinely Loyal'
             AND value_tier IN ('Gold', 'Platinum')
            THEN 'Stop Discounts'
            WHEN loyalty_type = 'Genuinely Loyal'
             AND value_tier IN ('Silver', 'Bronze')
            THEN 'Reduce Gradually'
            WHEN loyalty_type = 'Discount Dependent'
             AND value_tier IN ('Gold', 'Platinum')
            THEN 'Keep — High Risk'
            ELSE 'Monitor Only'
        END AS promo_action
    FROM customer_master
    GROUP BY loyalty_type, value_tier
    ORDER BY avg_spend DESC
""", engine)

promo_summary.to_csv(
    "powerbi_promo_summary.csv", index=False
)
print(f" powerbi_promo_summary.csv → {len(promo_summary)} rows")

# ── SAVE 5: Full Master ───────────────────────────
full_master = pd.read_sql_query(
    "SELECT * FROM customer_master", engine
)
full_master.to_csv(
    "powerbi_full_master.csv", index=False
)
print(f" powerbi_full_master.csv → {len(full_master)} rows")

# ── SAVE 6: ICP Summary ───────────────────────────
icp_data = {
    'attribute': [
        'Top Gender', 'Top Payment',
        'Top Category', 'Top Frequency',
        'Top Location'
    ],
    'value': [
        'Female', 'PayPal',
        'Clothing', 'Annually',
        'Texas'
    ]
}
icp_df = pd.DataFrame(icp_data)
icp_df.to_csv("powerbi_icp_summary.csv", index=False)
print(f" powerbi_icp_summary.csv → {len(icp_df)} rows")

print(f"""


ALL 5 BUSINESS QUESTIONS ANSWERED:
 Q1 → Loyal vs Discount customers
 Q2 → Behavioral patterns
 Q3 → Geographic analysis
 Q4 → Promo restructuring
 Q5 → Ideal customer profile

FILES SAVED FOR POWER BI:
 powerbi_loyalty_summary.csv
 powerbi_geo_summary.csv
 powerbi_category_summary.csv
 powerbi_promo_summary.csv
 powerbi_full_master.csv
 powerbi_icp_summary.csv

KEY FINDINGS SUMMARY:
→ 57% customers genuinely loyal 
→ 43% discount dependent 
→ Ideal customer: Female, PayPal,
  Clothing, Annual buyer, Texas
→ Golden Markets: 1,013 customers
→ Accessories = only retention category
→ Margin recovery possible from
  Genuinely Loyal + Gold/Platinum

NEXT → Week 4: Power BI Dashboard!
""")

Saving all results for Power BI...
 powerbi_loyalty_summary.csv → 8 rows
 powerbi_geo_summary.csv → 50 rows
 powerbi_category_summary.csv → 4 rows
 powerbi_promo_summary.csv → 8 rows
 powerbi_full_master.csv → 3900 rows
 powerbi_icp_summary.csv → 5 rows



ALL 5 BUSINESS QUESTIONS ANSWERED:
 Q1 → Loyal vs Discount customers
 Q2 → Behavioral patterns
 Q3 → Geographic analysis
 Q4 → Promo restructuring
 Q5 → Ideal customer profile

FILES SAVED FOR POWER BI:
 powerbi_loyalty_summary.csv
 powerbi_geo_summary.csv
 powerbi_category_summary.csv
 powerbi_promo_summary.csv
 powerbi_full_master.csv
 powerbi_icp_summary.csv

KEY FINDINGS SUMMARY:
→ 57% customers genuinely loyal 
→ 43% discount dependent 
→ Ideal customer: Female, PayPal,
  Clothing, Annual buyer, Texas
→ Golden Markets: 1,013 customers
→ Accessories = only retention category
→ Margin recovery possible from
  Genuinely Loyal + Gold/Platinum

NEXT → Week 4: Power BI Dashboard!



In [20]:
# Run this in Jupyter to get exact numbers
import pandas as pd

df = pd.read_csv("customer_master_final.csv")
df.columns = df.columns.str.strip().str.replace(' ', '_').str.lower()

print("NUMBERS FOR RETENTION PLAYBOOK:")
print("=" * 50)

# Total customers
total = len(df)
print(f"Total customers: {total}")

# Loyalty breakdown
print("\nLoyalty breakdown:")
print(df['loyalty_type'].value_counts())

# Value tier breakdown
print("\nValue tier:")
print(df['value_tier'].value_counts())

# Promo stats
disc_pct = (df['discount_applied'] == 'Yes').mean() * 100
print(f"\nDiscount usage: {disc_pct:.1f}%")

# Revenue stats
print(f"Total revenue: ${df['total_spent'].sum():,.2f}")
print(f"Avg spend: ${df['total_spent'].mean():.2f}")

# Champion customers
champions = df[
    (df['loyalty_type'] == 'Genuinely Loyal') &
    (df['value_tier'] == 'Platinum') &
    (df['satisfaction_flag'] == 'High')
]
print(f"\nChampion customers: {len(champions)}")
print(f"Champion avg spend: ${champions['total_spent'].mean():.2f}")

# Segments for sunset plan
print("\nGenuinely Loyal + Gold/Platinum:")
sunset = df[
    (df['loyalty_type'] == 'Genuinely Loyal') &
    (df['value_tier'].isin(['Gold', 'Platinum']))
]
print(f"Customers: {len(sunset)}")
print(f"Revenue: ${sunset['total_spent'].sum():,.2f}")
print(f"Margin recovery (15%): ${sunset['total_spent'].sum()*0.15:,.2f}")

# ICP details
print("\nIdeal customer details:")
print(f"Top gender: {df['gender'].value_counts().index[0]}")
print(f"Top payment: {df['payment_method'].value_counts().index[0]}")
print(f"Top category: {df['category'].value_counts().index[0]}")
print(f"Top location: {df['location'].value_counts().index[0]}")
print(f"Top frequency: {df['frequency_of_purchases'].value_counts().index[0]}")
print(f"Avg age: {df['age'].mean():.0f} years")
print(f"Avg prev purchases: {df['previous_purchases'].mean():.1f}")
print(f"Avg rating: {df['review_rating'].mean():.2f}")

NUMBERS FOR RETENTION PLAYBOOK:
Total customers: 3900

Loyalty breakdown:
loyalty_type
Genuinely Loyal       2223
Discount Dependent    1677
Name: count, dtype: int64

Value tier:
value_tier
Bronze      1014
Gold         988
Silver       972
Platinum     926
Name: count, dtype: int64

Discount usage: 43.0%
Total revenue: $233,081.00
Avg spend: $59.76

Champion customers: 286
Champion avg spend: $91.35

Genuinely Loyal + Gold/Platinum:
Customers: 1095
Revenue: $88,805.00
Margin recovery (15%): $13,320.75

Ideal customer details:
Top gender: Male
Top payment: Paypal
Top category: Clothing
Top location: Montana
Top frequency: Every 3 Months
Avg age: 44 years
Avg prev purchases: 25.4
Avg rating: 3.75


In [71]:
import os

playbook = """
RETENTION PLAYBOOK
SQL Customer Retention Strategy
D2C Fashion Brand — United States
════════════════════════════════════════════════════════

PREPARED BY : \ADARSH KUMAR
DATE        : June 2026
DATASET     : 3,900 customers | $233,081 total revenue
PURPOSE     : Reduce discount dependency without
              hurting sales volume

════════════════════════════════════════════════════════
SECTION 1 — EXECUTIVE SUMMARY
════════════════════════════════════════════════════════

The brand currently serves 3,900 customers across the
United States with total revenue of $233,081 and an
average purchase value of $59.76.

Analysis of customer behavioral data reveals:

KEY FINDING 1:
57% of customers (2,223) are Genuinely Loyal —
they purchase WITHOUT needing discounts.
This is a strong foundation that must be protected
and rewarded.

KEY FINDING 2:
43% of customers (1,677) are Discount Dependent —
they ONLY buy when a discount or promo code is applied.
This group costs the brand margin without building
long-term loyalty.

KEY FINDING 3:
43% of ALL purchases use discounts AND 43% use promo
codes — suggesting the same group uses both together.
The brand is training customers to wait for discounts
before purchasing.

KEY FINDING 4:
Accessories is the ONLY retention category — customers
who buy Accessories tend to be repeat purchasers.
All other categories (Clothing, Footwear, Outerwear)
are entry-point categories.

KEY FINDING 5:
1,013 customers live in Golden Market cities —
high spend + low promo dependency.
These are genuine brand advocates not yet targeted
by marketing.

THE CENTRAL QUESTION:
Is the brand building a loyal customer base or
relying on continuous promotional activity?

ANSWER:
The brand has a SPLIT reality:
57% genuine loyalty (healthy) vs
43% discount dependency (risky and growing).

Without intervention, the discount-dependent segment
will grow as more customers learn they can wait for
promotions — permanently reducing brand margins.

════════════════════════════════════════════════════════
SECTION 2 — CUSTOMER SEGMENTATION FINDINGS
════════════════════════════════════════════════════════

SEGMENT 1 — GENUINELY LOYAL (57% | 2,223 customers)
─────────────────────────────────────────────────────
Definition  : Promo dependency score = 0.0
              Used NO discount AND NO promo code
Behavior    : Buy at full price consistently
Avg spend   : Higher than discount dependent segment
Risk level  : LOW — will continue buying without promos
Action      : REWARD and PROTECT this segment

Sub-segments:
→ Platinum Loyal (545 customers)
   Highest spenders who never use discounts
   → STOP all discounts immediately
   → Enroll in exclusive loyalty program

→ Gold Loyal (550 customers)
   High spenders who never use discounts
   → STOP all discounts immediately
   → Offer loyalty points instead of discounts

→ Silver Loyal (560 customers)
   Mid spenders who never use discounts
   → REDUCE discounts by 50% over 3 months
   → Monitor purchase frequency carefully

→ Bronze Loyal (568 customers)
   Lower spenders who never use discounts
   → REDUCE discounts gradually
   → Focus on increasing basket size

────────────────────────────────────────────────────
SEGMENT 2 — MODERATE (uses ONE promo type)
─────────────────────────────────────────────────────
Definition  : Promo dependency score = 0.5
              Used EITHER discount OR promo code
              (not both)
Behavior    : Buy with some promotional support
Risk level  : MEDIUM — could go either way
Action      : CONVERT to genuine loyalty

Strategy:
→ Reward non-promo purchases with loyalty points
→ Send personalized recommendations (no discounts)
→ Create urgency through limited stock alerts
   rather than price discounts
→ Target with new Accessories products
   (retention category)

────────────────────────────────────────────────────
SEGMENT 3 — DISCOUNT DEPENDENT (43% | 1,677 customers)
─────────────────────────────────────────────────────
Definition  : Promo dependency score = 1.0
              Used BOTH discount AND promo code
Behavior    : Only purchases when promotions active
Risk level  : HIGH — currently hurting margins
Action      : GRADUAL reduction strategy

Sub-segments:
→ Platinum Discount Dependent (381 customers)
   High spenders BUT only with both promos
   → KEEP discounts for now (too risky to remove)
   → These are high value — cannot afford to lose
   → Slowly introduce loyalty program as alternative
   → Timeline: 6 months before any discount reduction

→ Gold Discount Dependent (438 customers)
   Good spenders but fully promo driven
   → KEEP discounts but cap at one type only
   → Remove either discount OR promo code (not both)
   → Timeline: 3 months

→ Silver Discount Dependent (412 customers)
   Mid spenders — promo driven
   → MONITOR only for 90 days
   → If they stop buying without promos → accept loss
   → Not worth high investment to convert

→ Bronze Discount Dependent (446 customers)
   Lowest spenders — bargain hunters
   → LOW PRIORITY
   → These are one-time bargain buyers
   → Do not invest in retaining this sub-segment

════════════════════════════════════════════════════════
SECTION 3 — PROMOTIONAL SUNSET PLAN
════════════════════════════════════════════════════════

WHAT IS A PROMO SUNSET PLAN?
Gradually reducing discounts for segments that do not
need them — protecting margin without losing volume.

PROMO SUNSET PLAN — 4 ACTIONS:
────────────────────────────────────────────────────

ACTION 1 — STOP DISCOUNTS (Immediate)
Target    : Genuinely Loyal + Gold + Platinum
Customers : 1,095 customers
Why       : These customers buy at full price already
            Discounts are unnecessary margin loss
When      : Start immediately — Month 1
How       : Remove from all discount email lists
            Remove promo code access
            Replace with loyalty points offer
Risk      : Very LOW — they already buy without promos
Metric    : Track purchase frequency monthly
            Success = no drop in purchase frequency
Expected  : Save 15% margin on $97,925 revenue
            = $14,689 margin recovery

ACTION 2 — REDUCE DISCOUNTS (3 months)
Target    : Genuinely Loyal + Silver + Bronze
Customers : 1,128 customers
Why       : These customers rarely use promos anyway
            Small reduction has minimal risk
When      : Month 2-3
How       : Reduce discount from 20% to 10%
            Then from 10% to 5%
            Then remove completely by Month 4
Risk      : LOW-MEDIUM
Metric    : Monitor conversion rate weekly
Success   : Conversion rate stays above 80% of baseline
Expected  : Partial margin recovery

ACTION 3 — KEEP DISCOUNTS (6 months)
Target    : Discount Dependent + Gold + Platinum
Customers : 819 customers
Why       : These are high value customers
            Removing discounts risks losing them
When      : Keep for 6 months then reassess
How       : Offer ONE type of discount (not both)
            Introduce loyalty program alongside
Risk      : HIGH if removed too soon
Metric    : Track if loyalty program enrollment grows
Success   : 30% enroll in loyalty program within 6 months

ACTION 4 — MONITOR ONLY
Target    : Discount Dependent + Silver + Bronze
Customers : 858 customers
Why       : Low value + high promo dependency
            Not worth investing heavily to retain
When      : Ongoing quarterly review
How       : Keep current promotions
            Do not create new promotions for this group
Risk      : LOW (already low value)
Metric    : Quarterly revenue contribution review

ROLLOUT TIMELINE:
─────────────────
Month 1   → Stop discounts for Loyal+Gold+Platinum
Month 2   → Reduce discounts for Loyal+Silver+Bronze
Month 3   → Launch loyalty program for all segments
Month 4   → Complete discount removal for Loyal segment
Month 5-6 → Assess Discount Dependent Gold+Platinum
Month 7+  → Full new promotional strategy in place

SUCCESS METRIC:
→ Primary   : Margin improvement % month over month
→ Secondary : Purchase frequency of Genuinely Loyal
→ Guard     : Overall revenue must not drop >5%

════════════════════════════════════════════════════════
SECTION 4 — IDEAL CUSTOMER PROFILE
════════════════════════════════════════════════════════

WHO IS THE BRAND'S BEST CUSTOMER?

Based on analysis of Champion customers:
(Genuinely Loyal + Platinum + High Satisfaction)

DEMOGRAPHIC PROFILE:
────────────────────
Gender          : Female
Age range       : 46-55 years (primary)
                  18-25 years (secondary — high growth)
Location        : Texas (primary market)
                  Montana (top city by customer count)

BEHAVIORAL PROFILE:
────────────────────
Payment method  : PayPal (preferred)
Purchase freq   : Annually (makes large purchases)
Avg purchase    : $91-95 per transaction
                  (above brand average of $59.76)
Previous buys   : 25+ purchases (very loyal!)
Review rating   : 3.8+ out of 5
Subscription    : YES — subscribed to brand

PRODUCT PROFILE:
────────────────────
Top category    : Clothing (Spring and Fall seasons)
Second category : Accessories (retention category!)
Top season      : Spring → Fall → Winter → Summer
Size preference : Varies (no dominant size)
Shipping pref   : Express (willing to pay for speed)

PSYCHOGRAPHIC PROFILE:
────────────────────
→ Values quality over price
→ Does NOT wait for discounts
→ Shops seasonally (Spring + Fall peaks)
→ Loyal to brand — 25+ previous purchases
→ Satisfied with brand experience (3.75+ rating)
→ Prefers convenience (PayPal + Express shipping)

WHAT MAKES THIS CUSTOMER VALUABLE:
────────────────────────────────────
→ Buys at full price → protects margins
→ Buys repeatedly → high lifetime value
→ High satisfaction → likely to refer others
→ Subscribed → engaged with brand

HOW TO ACQUIRE MORE IDEAL CUSTOMERS:
────────────────────────────────────
1. TARGET GEOGRAPHY:
   → Focus paid ads in Texas and Golden Market cities
   → These cities show high spend + low promo use
   → Genuine brand pull — not discount driven

2. TARGET DEMOGRAPHICS:
   → Female customers aged 45-55 on Instagram/Facebook
   → Female customers aged 18-25 on TikTok
   → These are your two highest-potential groups

3. TARGET PAYMENT METHOD:
   → Partner with PayPal for co-marketing
   → Offer PayPal exclusive early access (not discounts!)
   → PayPal users convert at higher rates

4. TARGET PRODUCT ENTRY POINT:
   → Acquire through Clothing (entry category)
   → Retain through Accessories (retention category)
   → New customer journey:
     Clothing purchase → Accessories upsell →
     Loyal repeat buyer

5. TARGET SEASON:
   → Launch acquisition campaigns in January-February
     (before Spring shopping season)
   → Spring is top season for ideal customers
   → Get them before competitors do

════════════════════════════════════════════════════════
SECTION 5 — GEOGRAPHIC STRATEGY
════════════════════════════════════════════════════════

FOUR MARKET TYPES IDENTIFIED:

GOLDEN MARKETS (1,013 customers — INVEST HERE!)
────────────────────────────────────────────────
Definition  : High avg spend + Low promo dependency
Meaning     : Customers in these cities genuinely
              love the brand — no discounts needed
Strategy    : Increase marketing spend in these cities
              Launch loyalty program first here
              No discount campaigns needed
Expected    : Highest ROI on marketing investment

PROMO MARKETS (1,005 customers — HANDLE CAREFULLY)
─────────────────────────────────────────────────────
Definition  : High avg spend + High promo dependency
Meaning     : Customers spend well BUT only with promos
Strategy    : Gradually reduce discount intensity
              Test loyalty program as alternative
              Do NOT remove discounts suddenly
              Risk of significant revenue loss if rushed

GROWTH MARKETS (792 customers — DEVELOP)
─────────────────────────────────────────
Definition  : Lower spend + Low promo dependency
Meaning     : Genuine interest but lower purchasing power
Strategy    : Introduce mid-range product line
              Grow basket size before reducing any promos
              Long term investment opportunity

LOW PRIORITY MARKETS (1,090 customers — MONITOR)
──────────────────────────────────────────────────
Definition  : Low spend + High promo dependency
Meaning     : Bargain hunters with low brand loyalty
Strategy    : Minimal investment
              Standard promotions only
              No special campaigns or loyalty investment

════════════════════════════════════════════════════════
SECTION 6 — CATEGORY STRATEGY
════════════════════════════════════════════════════════

CATEGORY FINDINGS:
──────────────────
Category     Type        Customers  Avg Prev Purchases
Accessories  Retention   1,240      25.7 (highest!)
Clothing     Entry       1,737      25.2
Footwear     Entry       599        25.2
Outerwear    Entry       324        25.0

KEY INSIGHT:
Accessories buyers have the HIGHEST previous purchase
count — meaning loyal customers gravitate toward
Accessories over time.

Clothing is the ENTRY point — most new customers
start with Clothing and then move to Accessories
as they become loyal.

CATEGORY RECOMMENDATIONS:
──────────────────────────
1. ACCESSORIES — Build retention category:
   → Expand Accessories product line
   → Create Accessories-specific loyalty rewards
   → Target existing Clothing buyers with
     Accessories upsell campaigns
   → Goal: Convert entry buyers to retention buyers

2. CLOTHING — Protect the entry point:
   → Keep Clothing well-stocked in all seasons
   → This is how most ideal customers ENTER
   → Spring Clothing campaigns = best acquisition tool
   → Do not over-discount Clothing — it trains
     customers to wait for sales

3. FOOTWEAR — Growth opportunity:
   → Currently small (599 customers)
   → Lower than average promo dependency (0.43)
   → Opportunity to grow without heavy discounting
   → Target existing Accessories buyers with
     Footwear recommendations

4. OUTERWEAR — Seasonal opportunity:
   → Smallest category (324 customers)
   → Focus on Fall/Winter seasonal campaigns
   → Bundle with Accessories for higher basket value

════════════════════════════════════════════════════════
SECTION 7 — 5 STRATEGIC RECOMMENDATIONS
════════════════════════════════════════════════════════

RECOMMENDATION 1 — LAUNCH LOYALTY PROGRAM
Priority   : HIGH — implement Month 3
Target     : All Genuinely Loyal customers (2,223)
What       : Points-based loyalty program
             Replace discounts with points
             Points redeemable for:
             → Early access to new collections
             → Free express shipping
             → Exclusive member-only products
             → NOT cash discounts
Why        : Builds genuine loyalty without
             training customers to expect discounts
Trade-off  : Initial setup cost vs long-term
             margin protection
Metric     : Loyalty program enrollment rate
Success    : 40% enrollment within 6 months

RECOMMENDATION 2 — STOP DISCOUNTS FOR TOP SEGMENT
Priority   : HIGH — implement Month 1
Target     : Genuinely Loyal + Gold + Platinum
             (1,095 customers)
What       : Remove from all discount lists
             immediately
Why        : Already buying at full price
             Discounts are pure margin loss
Trade-off  : Small risk of alienating some customers
             vs immediate margin recovery
Timeline   :
  Week 1   → Remove from discount email lists
  Week 2   → Remove promo code access
  Week 3   → Send loyalty program invite instead
  Week 4   → Monitor purchase rate closely
Metric     : Monthly purchase frequency
Success    : No more than 5% drop in purchase rate
Expected   : $14,689 margin recovery (estimated)

RECOMMENDATION 3 — TARGET GOLDEN MARKET CITIES
Priority   : MEDIUM — implement Month 2
Target     : Golden Market cities (1,013 customers)
What       : Increase paid advertising budget in
             Golden Market cities by 30%
             Focus on:
             → Instagram/Facebook for 45-55 Female
             → TikTok for 18-25 Female
             → PayPal partnership campaigns
Why        : These cities have high spend + low promo
             dependency — best ROI on marketing
Trade-off  : Higher marketing spend upfront vs
             acquiring ideal customer profile
Metric     : New customer acquisition cost (CAC)
             in Golden Market cities
Success    : CAC in Golden Markets below brand average

RECOMMENDATION 4 — ACCESSORIES EXPANSION
Priority   : MEDIUM — implement Month 3-4
Target     : All existing Clothing buyers
What       : Upsell Accessories to Clothing customers
             → Post-purchase email:
               "Customers like you also love..."
             → Loyalty points for first Accessories buy
             → Bundle: Clothing + Accessories discount
               for FIRST-TIME Accessories purchase only
Why        : Accessories = retention category
             Moving customers from Clothing to Accessories
             increases lifetime value significantly
Trade-off  : Small discount on first Accessories
             purchase vs long-term retention gain
Metric     : % of Clothing buyers who buy Accessories
Success    : 15% conversion from Clothing to Accessories
             within 6 months

RECOMMENDATION 5 — SPRING ACQUISITION CAMPAIGN
Priority   : MEDIUM — implement before next Spring
Target     : New customer acquisition
What       : Major Spring campaign targeting
             Female customers in Golden Market cities
             Focus on Clothing (entry category)
             NO discount — use:
             → "New Spring Collection" urgency
             → Free shipping threshold
             → Style guide content marketing
Why        : Spring = top season for ideal customers
             Clothing = how ideal customers enter brand
             Golden Markets = genuine brand pull exists
Trade-off  : Campaign spend vs customer acquisition
Metric     : New customers acquired in Spring campaign
Success    : 10% increase in new customers vs last Spring

════════════════════════════════════════════════════════
SECTION 8 — EXPECTED BUSINESS IMPACT
════════════════════════════════════════════════════════

IF ALL RECOMMENDATIONS IMPLEMENTED:

SHORT TERM (0-3 months):
→ Margin recovery from stopping discounts:
  ~$14,689 (estimated 15% on Loyal+Gold+Platinum)
→ No expected revenue drop from loyal segment
→ Potential 5% drop from discount segment
  (acceptable and expected)

MEDIUM TERM (3-6 months):
→ Loyalty program enrollment builds repeat rates
→ Golden Market acquisition brings ideal customers
→ Accessories expansion improves lifetime value
→ Expected: 8-12% improvement in margin %

LONG TERM (6-12 months):
→ Brand no longer dependent on promotions
→ Loyal customer base grows as % of total
→ Discount dependent segment shrinks naturally
→ Expected: 15-20% margin improvement
            without revenue loss

RISKS TO MONITOR:
→ Competitor runs heavy discounting campaign
  → Counter with loyalty program benefits NOT price
→ Discount segment churns faster than expected
  → Increase Golden Market acquisition speed
→ Loyalty program adoption is slow
  → Simplify program and increase awareness

════════════════════════════════════════════════════════
SECTION 9 — METRICS DASHBOARD
════════════════════════════════════════════════════════

TRACK THESE METRICS MONTHLY:

LOYALTY HEALTH:
→ % Genuinely Loyal customers (target: grow from 57%)
→ % Discount Dependent customers (target: shrink from 43%)
→ Loyalty program enrollment rate (target: 40%)

REVENUE HEALTH:
→ Total revenue (guard: must not drop >5%)
→ Avg purchase value (target: grow from $59.76)
→ Revenue from non-discounted purchases (target: grow)

ACQUISITION HEALTH:
→ New customers from Golden Market cities
→ % new customers who become repeat buyers
→ Customer Acquisition Cost by city type

RETENTION HEALTH:
→ Purchase frequency of Genuinely Loyal segment
→ % Clothing buyers who buy Accessories
→ Subscription program retention rate

════════════════════════════════════════════════════════
END OF RETENTION PLAYBOOK
════════════════════════════════════════════════════════

Document prepared by: Adarsh Kumar
Project: SQL Customer Retention Analysis
Organization: Consulting & Analytics Club — IIT Guwahati
Date: June 2026
"""

# Save the playbook
with open("Retention_Playbook.txt",
          "w", encoding="utf-8") as f:
    f.write(playbook)

print(" Retention_Playbook.txt saved!")
print(f"   Location: {os.getcwd()}")
print(f"   Length: {len(playbook.split())} words")

 Retention_Playbook.txt saved!
   Location: C:\Users\BIT\Desktop\sql project
   Length: 2676 words
